# Pattern 1: Direct SDK Access

The simplest pattern — IAM-authenticated SDK calls to create a managed Knowledge Base,
ingest S3 documents, and retrieve results. No Gateway layer.

**What you get:** Basic RAG pipeline with IAM authentication.

**What you don't get:** Fine-grained access control, per-user scoping, or centralized auth.

## Prerequisites

- **boto3 ≥ 1.42.78** for `bedrock-agent` (KB creation, data source, ingestion)
- **boto3 ≥ 1.42.69** for `bedrock-agent-runtime` (retrieval with `managedSearchConfiguration`)
- IAM role with Bedrock KB execution permissions
- S3 bucket with documents to ingest

## Architecture

```
Client (IAM) ──► Bedrock Agent API ──► Managed KB ──► S3 Data Source
```

In [ ]:
import sys
import boto3
import time
import json
import util   


# --- Configuration (update these) ---
REGION = "us-west-2"
S3_BUCKET = "<existing-or-unique-name-for-your-kb-bucket->"     
S3_PREFIX = "documents/"  # objects outside of the prefix will not be indexed

session = boto3.Session()

ROLE_ARN = None   # Replace with your own ARN. if none exists it will create one
     
if ROLE_ARN is None:
    info = util.setup(bucket_name=S3_BUCKET, prefix=S3_PREFIX, region_name=REGION)
    ROLE_ARN = info["role_arn"]
    S3_BUCKET = info['bucket']
    S3_PREFIX = info['prefix']

# e.g. "arn:aws:bedrock:us-west-2::foundation-model/amazon.titan-embed-text-v2:0"
EMBEDDING_MODEL = "<your-embedding-model-arn>" 
GENERATION_MODEL = "<your-generation-model-arn>"  # e.g. inference profile ARN for Claude

# Clients
session = boto3.Session()
cp = session.client("bedrock-agent", region_name=REGION)
dp = session.client("bedrock-agent-runtime", region_name=REGION)

S3_ACCOUNT = session.client("sts").get_caller_identity()["Account"]

print(f"boto3 {boto3.__version__}")


In [ ]:
# Create a MANAGED Knowledge Base using the managed DEFAULT embedding model

response = cp.create_knowledge_base(
    name=f"p1-direct-{int(time.time())}",
    roleArn=ROLE_ARN,
    knowledgeBaseConfiguration={
        "type": "MANAGED",
        "managedKnowledgeBaseConfiguration": {}   # empty = managed default embedding
    }
)

kb_id = response["knowledgeBase"]["knowledgeBaseId"]
print(f"KB created: {kb_id}")

for _ in range(30):
    status = cp.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"]
    if status == "ACTIVE":
        break
    time.sleep(5)
print(f"KB status: {status}")


In [ ]:
# Create S3 data source using MANAGED_KNOWLEDGE_BASE_CONNECTOR
response = cp.create_data_source(
    knowledgeBaseId=kb_id,
    name="s3-source",
    dataSourceConfiguration={
        "type": "MANAGED_KNOWLEDGE_BASE_CONNECTOR",
        "managedKnowledgeBaseConnectorConfiguration": {
            "connectorParameters": {
                "type": "S3",
                "version": "1",
                "connectionConfiguration": {
                    "bucketName": S3_BUCKET,
                    "bucketOwnerAccountId": S3_ACCOUNT
                },
                "filterConfiguration": {
                    "inclusionPrefixes": [S3_PREFIX]
                },
                "deletionProtectionConfiguration": {
                    "enableDeletionProtection": False
                }
            },
            "deletionProtectionConfiguration": {
                "deletionProtectionStatus": "DISABLED"
            }
        }
    },
    vectorIngestionConfiguration={
        "parsingConfiguration": {"parsingStrategy": "SMART_PARSING"}
    }
)

ds_id = response["dataSource"]["dataSourceId"]
print(f"Data source created: {ds_id}")

# Wait for AVAILABLE
for _ in range(12):
    status = cp.get_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)["dataSource"]["status"]
    if status == "AVAILABLE":
        break
    time.sleep(5)
print(f"Data source status: {status}")

In [ ]:
# Start ingestion job — crawls S3 and indexes documents
response = cp.start_ingestion_job(
    knowledgeBaseId=kb_id,
    dataSourceId=ds_id
)

job_id = response["ingestionJob"]["ingestionJobId"]
print(f"Ingestion job: {job_id}")

# Poll until COMPLETE or FAILED
for _ in range(40):
    job = cp.get_ingestion_job(
        knowledgeBaseId=kb_id,
        dataSourceId=ds_id,
        ingestionJobId=job_id
    )["ingestionJob"]
    if job["status"] in ("COMPLETE", "FAILED"):
        break
    time.sleep(15)

stats = job.get("statistics", {})
print(f"Status: {job['status']}")
print(f"  Scanned:  {stats.get('numberOfDocumentsScanned', 0)}")
print(f"  Indexed:  {stats.get('numberOfNewDocumentsIndexed', 0)}")
print(f"  Failed:   {stats.get('numberOfDocumentsFailed', 0)}")

## Two Retrieval APIs

| API | Returns | Use When |
|---|---|---|
| **Retrieve** | Raw chunks with scores | You want source chunks for a custom RAG pipeline |
| **AgenticRetrieveStream** | Deduplicated chunks (streaming) | Complex/multi-part queries, multi-KB retrieval |

In [ ]:
# --- Retrieve API ---
# Returns raw source chunks with relevance scores
QUESTION = "What regulatory compliance risks does the company face?"

response = dp.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={"text": QUESTION},
    retrievalConfiguration={
        "managedSearchConfiguration": {
            "numberOfResults": 5
        }
    }
)

print("=== Retrieve API ===")
for i, res in enumerate(response.get("retrievalResults", []), 1):
    score = res["score"]
    text = res["content"]["text"][:100]
    loc_type = res.get("location", {}).get("type", "")
    uri = res.get("location", {}).get("s3Location", {}).get("uri", "N/A")
    print(f"{i}. score={score:.4f} | {text}...")
    print(f"   source: {uri.split('/')[-1]}")
print(f"\nTotal: {len(response.get('retrievalResults', []))} chunks")

In [ ]:
# --- Generate separately from retrieved chunks ---
# The Retrieve cell above gave us `response` with source chunks.
# Here we build a context from them and ask the model to answer — classic RAG,
# with retrieval and generation as two explicit steps.
# if verbose is True also print the prompt 

verbose = True

# 1. Collect the retrieved chunks as context
chunks = response.get("retrievalResults", [])
context = "\n\n".join(
    f"[{i}] {res['content']['text']}"
    for i, res in enumerate(chunks, 1)
)

# 2. Build the prompt
prompt = (
    "Answer the question using only the context below. "
    "Cite sources by their [number]. If the answer isn't in the context, say so.\n\n"
    f"Context:\n{context}\n\n"
    f"Question: {QUESTION}"
)

# 3. Call the generation model via the Converse API
brt = session.client("bedrock-runtime", region_name=REGION)
gen = brt.converse(
    modelId=GENERATION_MODEL,   # inference-profile ARN works here
    messages=[{"role": "user", "content": [{"text": prompt}]}],
    inferenceConfig={"maxTokens": 1000, "temperature": 0.0},
)


if verbose:
    print("=== Prompt sent to model ===")
    print(prompt)
    print("=" * 40)

answer = gen["output"]["message"]["content"][0]["text"]
print("=== Generated Answer ===")
print(answer)
print(f"\n(from {len(chunks)} retrieved chunks)")


In [ ]:
# --- AgenticRetrieveStream API ---
# Decomposes complex queries into sub-queries, runs multiple retrievals,
# and returns deduplicated chunks via streaming

response = dp.agentic_retrieve_stream(
    messages=[
        {
            "role": "user",
            "content": {
                "text": "What are the three U.S. regions where tornadoes form most frequently, and what states are included in each?"
            }
        }
    ],
    retrievers=[
        {
            "configuration": {
                "knowledgeBase": {
                    "knowledgeBaseId": kb_id,
                    "retrievalOverrides": {
                        "maxNumberOfResults": 10
                    }
                }
            }
        }
    ],
    agenticRetrieveConfiguration={
        "foundationModelConfiguration": {
            "type": "BEDROCK_FOUNDATION_MODEL",
            "bedrockFoundationModelConfiguration": {
                "modelConfiguration": {"modelArn": GENERATION_MODEL}
            },
        },
        "foundationModelType": "CUSTOM",
        "maxAgentIteration": 3,
        "rerankingModelType": "MANAGED",
    }
)

print("=== AgenticRetrieveStream API ===")
trace_count = 0
final_results = []

for event in response["stream"]:
    if "traceEvent" in event:
        attrs = event["traceEvent"].get("attributes", {})
        step = attrs.get("step", "unknown")
        status = attrs.get("status", "")
        print(f"  [trace] step={step} status={status}")
        trace_count += 1
    elif "result" in event:
        final_results = event["result"].get("results", [])
        print(f"\nFinal: {len(final_results)} deduplicated chunks")
        for i, r in enumerate(final_results[:5], 1):
            retriever = r.get("sourceRetriever", {}).get("identifier", "N/A")
            text = r["content"]["text"][:100]
            print(f"  {i}. retriever={retriever} | {text}...")

print(f"\nTrace events: {trace_count} | Final chunks: {len(final_results)}")

### Key Differences

- **Retrieve**: `managedSearchConfiguration` — returns raw chunks, you build the prompt (then generate separately, e.g. via the Converse API)
- **AgenticRetrieveStream**: `messages` + `retrievers` — streaming, multi-KB, query decomposition; set `rerankingModelType="MANAGED"` (default-embedding KBs) or `"NONE"` (custom-embedding KBs)

> **Note:** `messages[0].content` in AgenticRetrieveStream is a struct `{"text": "..."}`, not a list.
> The `s3Location.uri` in results is an HTTPS URL, not an `s3://` URI.


In [ ]:
# Cleanup — delete data source first, then KB
cp.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)
cp.delete_knowledge_base(knowledgeBaseId=kb_id)
print(f"Deleted KB {kb_id} and data source {ds_id}")

## Summary

**Pattern 1** gives you the simplest RAG pipeline with all two retrieval APIs:

| API | Use Case |
|---|---|
|`Retrieve` | Custom RAG — get chunks, then generate separately (e.g. Converse API) |
| `AgenticRetrieveStream` | Complex queries — multi-KB, query decomposition, streaming |

| Feature | Status |
|---|---|
| Authentication | IAM only |
| Authorization | IAM policies |
| Per-user scoping | ❌ |
| KB ID hidden from agents | ❌ |
| Centralized access control | ❌ |
| Audit logging | CloudTrail only |

**When to use:** Internal tools, prototypes, single-tenant apps where IAM is sufficient.

**Next:** [Pattern 2](02-metadata-filters.ipynb) adds document-level scoping with metadata filters.